# 2층 신경망 순전파·역전파 손계산 검증

**구조:** 입력 2개 → 은닉 2개 → 출력 1개, 두 층 모두 Sigmoid, 손실함수 BCE.  
**가정:** 과제에서 `y_true`가 빈칸이므로 `y_true = 1`로 설정.  
**행벡터 표기:** $z_1=xW_1+b_1$, $z_2=a_1\cdot W_2+b_2$. `x.shape == (2,)`, `W1.shape == (2,2)`.

순전파: $z_1=xW_1+b_1$, $a_1=\sigma(z_1)$, $z_2=a_1\cdot W_2+b_2$, $\hat y=\sigma(z_2)$.  
손실: $L=-[y\ln\hat y+(1-y)\ln(1-\hat y)]$.

역전파: $\partial L/\partial\hat y=-y/\hat y+(1-y)/(1-\hat y)$, $\delta_2=\partial L/\partial z_2=\hat y-y$, $\partial L/\partial W_2=\delta_2 a_1$, $\partial L/\partial a_1=\delta_2W_2$, $\delta_1=\partial L/\partial z_1=(\partial L/\partial a_1)\odot a_1\odot(1-a_1)$, $\partial L/\partial W_1=x^T\delta_1$ (외적).

In [1]:
import numpy as np

x = np.array([1.0, 0.0])
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([0.0, 0.0])
W2 = np.array([0.5, 0.6])
b2 = 0.0
y_true = 1.0

def sigmoid(z):
    """입력 스칼라/배열에 Sigmoid를 원소별 적용한다."""
    return 1 / (1 + np.exp(-z))

# 순전파
z1 = x @ W1 + b1
a1 = sigmoid(z1)
z2 = a1 @ W2 + b2
y_pred = sigmoid(z2)
loss = -(y_true * np.log(y_pred) + (1-y_true) * np.log(1-y_pred))

# 역전파: 중간 기울기 재사용
dL_dy_pred = -y_true / y_pred + (1-y_true) / (1-y_pred)
dy_pred_dz2 = y_pred * (1-y_pred)
delta2 = dL_dy_pred * dy_pred_dz2  # = y_pred - y_true
dW2 = delta2 * a1
db2 = delta2
da1 = delta2 * W2
delta1 = da1 * a1 * (1-a1)
dW1 = np.outer(x, delta1)  # x[:, None] @ delta1[None, :]와 동일
db1 = delta1

for name, value in [
    ('z1',z1), ('a1',a1), ('z2',z2), ('y_pred',y_pred), ('L',loss),
    ('dL/dy_pred',dL_dy_pred), ('dy_pred/dz2',dy_pred_dz2),
    ('delta2 = dL/dz2',delta2), ('dL/dW2',dW2),
    ('dL/da1',da1), ('delta1 = dL/dz1',delta1), ('dL/dW1',dW1),
    ('dL/db2',db2), ('dL/db1',db1)
]:
    print(f'{name:21s} = {np.array2string(np.asarray(value), precision=4, floatmode="fixed")}   shape={np.shape(value)}')

z1                    = [0.1000 0.2000]   shape=(2,)
a1                    = [0.5250 0.5498]   shape=(2,)
z2                    = 0.5924   shape=()
y_pred                = 0.6439   shape=()
L                     = 0.4402   shape=()
dL/dy_pred            = -1.5530   shape=()
dy_pred/dz2           = 0.2293   shape=()
delta2 = dL/dz2       = -0.3561   shape=()
dL/dW2                = [-0.1869 -0.1958]   shape=(2,)
dL/da1                = [-0.1780 -0.2137]   shape=(2,)
delta1 = dL/dz1       = [-0.0444 -0.0529]   shape=(2,)
dL/dW1                = [[-0.0444 -0.0529]
 [-0.0000 -0.0000]]   shape=(2, 2)
dL/db2                = -0.3561   shape=()
dL/db1                = [-0.0444 -0.0529]   shape=(2,)


## 손계산 결과와 4자리 비교

손계산 기준: `z1=[0.1,0.2]`, `a1=[0.5250,0.5498]`, `z2=0.5924`, `y_pred=0.6439`, `L=0.4402`, `dL/dy_pred=-1.5530`, `delta2=-0.3561`, `dW2=[-0.1869,-0.1958]`, `da1=[-0.1780,-0.2137]`, `delta1=[-0.0444,-0.0529]`, `dW1=[[-0.0444,-0.0529],[0,0]]`.

In [2]:
expected = {
    'z1': np.array([0.1000, 0.2000]),
    'a1': np.array([0.5250, 0.5498]),
    'z2': 0.5924,
    'y_pred': 0.6439,
    'L': 0.4402,
    'dL/dy_pred': -1.5530,
    'delta2': -0.3561,
    'dW2': np.array([-0.1869, -0.1958]),
    'da1': np.array([-0.1780, -0.2137]),
    'delta1': np.array([-0.0444, -0.0529]),
    'dW1': np.array([[-0.0444, -0.0529], [0.0000, 0.0000]]),
}
actual = {
    'z1': z1, 'a1': a1, 'z2': z2, 'y_pred': y_pred, 'L': loss,
    'dL/dy_pred': dL_dy_pred, 'delta2': delta2, 'dW2': dW2,
    'da1': da1, 'delta1': delta1, 'dW1': dW1,
}
for name, hand in expected.items():
    computed = np.round(np.asarray(actual[name]), 4)
    assert np.array_equal(computed, hand), f'{name}: 검증 실패! {computed} != {hand}'
    print(f'{name}: 4자리 일치 ✓')
print('모든 항목 손계산과 소수점 4자리까지 일치!')

z1: 4자리 일치 ✓
a1: 4자리 일치 ✓
z2: 4자리 일치 ✓
y_pred: 4자리 일치 ✓
L: 4자리 일치 ✓
dL/dy_pred: 4자리 일치 ✓
delta2: 4자리 일치 ✓
dW2: 4자리 일치 ✓
da1: 4자리 일치 ✓
delta1: 4자리 일치 ✓
dW1: 4자리 일치 ✓
모든 항목 손계산과 소수점 4자리까지 일치!
